# EFG point-charge

Here we reproduce results discussed [here](https://github.com/apdioguardi/EFG_point_charge_lattice_sum/tree/main/EFG_point_charge_aAlO2).

In [1]:
import numpy as np
from ase import Atoms
from ase.io import read
from pymatgen.core import Structure
from pymatgen.io.ase import AseAtomsAdaptor

In [2]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent.parent.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT)) 

from pcefg.src.pcefg import constants
from pcefg.src.pcefg.lattice import get_atom_kinds
from pcefg.src.pcefg.point_charge import PointChargeEFG
from pcefg.src.pcefg.point_charge import (
    compute_efg,
    diagonalize_EFG,
    point_charge_EFG,
    sphere_radius_convergence
)

In [3]:
import os
dpath='./'

In [4]:
file = "Ishizawa_1980_aAl2O3_300K_EntryWithCollCode10425.cif"  # 'read_cif' ERROR: CIF File not structured well, converted to VASP format below
file = "Ishizawa_1980_aAl2O3_300K_EntryWithCollCode10425.vasp"
file = os.path.join(dpath, file)
st = Structure.from_file(file)
print(st)
# atoms = AseAtomsAdaptor.get_atoms(st) # ISSUE: AseAtomsAdaptor discard distinct atoms
atoms = read(file)

Full Formula (Al12 O18)
Reduced Formula: Al2O3
abc   :   4.754000   4.754000  12.990000
angles:  90.000000  90.000000 120.000000
pbc   :       True       True       True
Sites (30)
  #  SP           a         b         c
---  ----  --------  --------  --------
  0  Al    0         0         0.35228
  1  Al    0         0         0.64772
  2  Al    0         0         0.14772
  3  Al    0         0         0.85228
  4  Al    0.666667  0.333333  0.685613
  5  Al    0.666667  0.333333  0.981053
  6  Al    0.666667  0.333333  0.481053
  7  Al    0.666667  0.333333  0.185613
  8  Al    0.333333  0.666667  0.018947
  9  Al    0.333333  0.666667  0.314387
 10  Al    0.333333  0.666667  0.814387
 11  Al    0.333333  0.666667  0.518947
 12  O     0.3064    0         0.25
 13  O     0.6936    0         0.75
 14  O     0         0.3064    0.25
 15  O     0         0.6936    0.75
 16  O     0.6936    0.6936    0.25
 17  O     0.3064    0.3064    0.75
 18  O     0.973067  0.333333  0.583333
 19  O 

In [5]:
gamma_sternheimer = {  # or gamma_inf, anti-shielding factors, for the ions in your material
    'Na': -4.0,
    'O':  -2.2
    # 'Mn': ...,  # add your own value here if you have one
}

# Formal ionic charges [units of e], only needed if compute_lattice_efg=True.
formal_charges = {
    'Na': +1, 'F': -1, 'Ca': +2, 'Li': +1, 'Cl': -1, 'Mg': +2, 'O': -2,
    'Al': +3,
    # add whatever your material contains
}

In [6]:
# Identify atom kinds

atm_kinds = get_atom_kinds(atoms)

print("\nAtom kinds:")
for kind, indices in atm_kinds.items():
    print(f"{kind}: {indices}")


print("\nO indices:")
print(atm_kinds["O"])


Atom kinds:
Al: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
O: [12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]

O indices:
[12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]


In [7]:
# O_idx   = [i for i, s in enumerate(atoms.get_chemical_symbols()) if s == 'O']
# Al_idx  = [i for i, s in enumerate(atoms.get_chemical_symbols()) if s == 'Al']

# test non zero spin O isotopes (e.g., 17^O, I=5/2, Q=-0.0265e-28 m^2, 0.038% abundance)
O_nuclear_spin   = 5/2
O_Quadrupole_moment = -0.0265e-28  # m^2 (0.038 abundance)


# test non zero spin Al isotopes (e.g., 27^Al, I=5/2, Q=+0.1466e-28 m^2, 100% abundance)
Al_I_spin   = 5/2
Al_Q_moment = +0.1466e-28  # m^2 (100 abundance)

Al2O3_charges = {'Al': +3, 'O': -2}

sphere_radius=50

Sternheimer antishielding factor = 0.0

In [8]:
probe_idx = atm_kinds["O"][4]
probe_pos = atoms.get_scaled_positions()[probe_idx]  

res = compute_efg(
    atoms, 
    probe_position=probe_pos, 
    atomic_charges=Al2O3_charges, 
    sphere_radius=sphere_radius,
    gamma_sternheimer=-0.0, 
    exclude_indices=(), 
    extra_charges=None,
    coords_are_cartesian=False, # True or False, depending on 'probe_position' type, frac. or cart.
    nuclear_spin=O_nuclear_spin,
    quadrupole_moment=O_Quadrupole_moment,
    verbose=True
)

Point-charge EFG: summing 61800 charges within radius 50.000 Å.

EFG analysis for probe site at frac coord. (0.6936, 0.6936, 0.2500)
Vzz          = -1.17942258e+21 V/m^2
Vyy          =  9.52937763e+20 V/m^2
Vxx          =  2.26484815e+20 V/m^2
eta          =  0.61593950 (unitless)
chi_Q_MHz    =  0.75573524 MHz
nu_z_MHz     =  0.11336029 MHz
nu_Q_MHz     =  0.12031476 MHz

EFG tensor V_ab (V/m^2) =
----------------------------------------------------------------------
 [ -7.68149844e+18,  1.35195984e+20, -9.23031800e+20 ]
 [  1.35195984e+20,  1.48429377e+20,  5.32912658e+20 ]
 [ -9.23031800e+20,  5.32912658e+20, -1.40747879e+20 ]
----------------------------------------------------------------------
Trace(V_ab) =  4.09600e+05
Symmetric   = True

principal axes (unitless) = 
----------------------------------------------------------------------
 [ -5.00000000e-01,  6.20221174e-01, -6.04421786e-01 ]
 [ -8.66025404e-01, -3.58084862e-01,  3.48963081e-01 ]
 [ -7.52786722e-12, -6.97926162e-0

Sternheimer antishielding factor = -2.2

In [9]:
probe_idx = atm_kinds["O"][4]
probe_pos = atoms.get_scaled_positions()[probe_idx]  
probe_pos = atoms.get_positions()[probe_idx] 

res = compute_efg(
    atoms, 
    probe_position=probe_pos, 
    atomic_charges=Al2O3_charges, 
    sphere_radius=sphere_radius,
    gamma_sternheimer=-2.2, 
    exclude_indices=(), 
    extra_charges=None,
    coords_are_cartesian=True, # True or False, depending on 'probe_position' type, frac. or cart.
    nuclear_spin=O_nuclear_spin,
    quadrupole_moment=O_Quadrupole_moment,
    verbose=True
)

Point-charge EFG: summing 61800 charges within radius 50.000 Å.

EFG analysis for probe site at frac coord. (0.6936, 0.6936, 0.2500)
Vzz          = -3.77415225e+21 V/m^2
Vyy          =  3.04940084e+21 V/m^2
Vxx          =  7.24751407e+20 V/m^2
eta          =  0.61593950 (unitless)
chi_Q_MHz    =  2.41835278 MHz
nu_z_MHz     =  0.36275292 MHz
nu_Q_MHz     =  0.38500724 MHz

EFG tensor V_ab (V/m^2) =
----------------------------------------------------------------------
 [ -2.45807950e+19,  4.32627148e+20, -2.95370176e+21 ]
 [  4.32627148e+20,  4.74974006e+20,  1.70532051e+21 ]
 [ -2.95370176e+21,  1.70532051e+21, -4.50393211e+20 ]
----------------------------------------------------------------------
Trace(V_ab) =  1.24518e+06
Symmetric   = True

principal axes (unitless) = 
----------------------------------------------------------------------
 [ -5.00000000e-01,  6.20221174e-01, -6.04421786e-01 ]
 [ -8.66025404e-01, -3.58084862e-01,  3.48963081e-01 ]
 [ -7.52775620e-12, -6.97926162e-0